# Fichiers modernes — `pathlib`, `csv` et `json`

Le module « fichiers » a montré `open()` et `os.path`. Ici, la version **moderne et recommandée** :

- **`pathlib`** : manipuler les chemins comme des **objets** au lieu de bricoler des strings.
  C'est l'API recommandée depuis Python 3.4 — `os.path` reste correct mais date d'avant.
- **`csv`** : lire/écrire du CSV **sans se faire piéger** (le `split(",")` maison casse dès qu'une
  valeur contient une virgule !).
- **`json`** : LE format d'échange de données (APIs, fichiers de config, sauvegardes).

---
## 1. `pathlib.Path` — le chemin devient un objet

Un `Path` représente un chemin de fichier ou de dossier. Ses gros avantages :

1. L'opérateur `/` assemble les chemins — fini les `os.path.join` et les `\\` vs `/`
   (Path gère la différence Windows/Linux tout seul).
2. Tout est accroché à l'objet : existence, taille, extension, lecture, écriture...

| Ancien style (`os.path`) | Style moderne (`pathlib`) |
|---|---|
| `os.path.join(dossier, "a.txt")` | `dossier / "a.txt"` |
| `os.path.exists(p)` | `p.exists()` |
| `os.path.splitext(p)[1]` | `p.suffix` |
| `os.path.basename(p)` | `p.name` |
| `os.listdir(d)` | `d.iterdir()` ou `d.glob("*.txt")` |
| `open(p).read()` | `p.read_text(encoding="utf-8")` |

In [ ]:
from pathlib import Path

# On travaille dans un sous-dossier de démo (créé à côté du notebook)
dossier = Path("demo_fichiers")
dossier.mkdir(exist_ok=True)          # crée le dossier ; exist_ok=True → pas d'erreur s'il existe

# Assembler un chemin avec / — lisible et portable (Windows, Linux, Mac)
fichier = dossier / "notes.txt"

# Écrire et lire en une ligne (ouverture + fermeture gérées automatiquement)
fichier.write_text("Première ligne\nDeuxième ligne\n", encoding="utf-8")
contenu = fichier.read_text(encoding="utf-8")
print(contenu)

# Tout est accessible sur l'objet :
print(f"Existe      : {fichier.exists()}")
print(f"Nom         : {fichier.name}")        # notes.txt
print(f"Extension   : {fichier.suffix}")      # .txt
print(f"Sans ext.   : {fichier.stem}")        # notes
print(f"Dossier     : {fichier.parent}")      # demo_fichiers
print(f"Taille      : {fichier.stat().st_size} bytes")
print(f"Absolu      : {fichier.resolve()}")

In [ ]:
from pathlib import Path

dossier = Path("demo_fichiers")

# Créons quelques fichiers pour la démonstration
(dossier / "rapport_janvier.csv").write_text("a;b\n1;2\n", encoding="utf-8")
(dossier / "rapport_fevrier.csv").write_text("a;b\n3;4\n", encoding="utf-8")
(dossier / "lisez_moi.md").write_text("# Infos\n", encoding="utf-8")

# glob : chercher par motif (* = n'importe quoi). rglob = pareil, mais dans les sous-dossiers aussi.
print("Tous les CSV :")
for chemin in sorted(dossier.glob("*.csv")):
    print(f"  {chemin.name}  ({chemin.stat().st_size} bytes)")

# glob retourne un générateur : parfait pour parcourir des milliers de fichiers sans liste en mémoire
nb_fichiers = sum(1 for _ in dossier.iterdir())
print(f"\n{nb_fichiers} éléments dans {dossier}/")

---
## 2. Le module `csv` — pourquoi pas `split(",")` ?

Un CSV (*Comma-Separated Values* : valeurs séparées par des virgules) a l'air simple...
jusqu'à ce qu'une valeur contienne elle-même une virgule :

```
nom,description,prix
Clavier,"Compact, sans fil",49.99   ← le split(",") coupe la description en deux !
```

Le module `csv` gère correctement les guillemets, les séparateurs et les retours à la ligne.
En pratique, on utilise surtout `DictReader`/`DictWriter` : chaque ligne devient un **dictionnaire**
dont les clés sont les en-têtes de colonnes — le code reste lisible même avec 30 colonnes.

In [ ]:
import csv
from pathlib import Path

fichier_csv = Path("demo_fichiers") / "produits.csv"

produits = [
    {"nom": "Clavier", "description": "Compact, sans fil", "prix": "49.99"},
    {"nom": "Écran", "description": 'Dalle 27" IPS', "prix": "219.00"},
    {"nom": "Souris", "description": "Ergonomique", "prix": "19.99"},
]

# ── Écriture ──  newline="" est OBLIGATOIRE avec csv (sinon lignes vides parasites sous Windows)
with open(fichier_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["nom", "description", "prix"])
    writer.writeheader()
    writer.writerows(produits)

print(fichier_csv.read_text(encoding="utf-8"))
# Remarque : le module a mis des guillemets tout seul là où c'était nécessaire

# ── Lecture ──  chaque ligne arrive en dictionnaire {colonne: valeur}
with open(fichier_csv, encoding="utf-8", newline="") as f:
    for ligne in csv.DictReader(f):
        prix = float(ligne["prix"])          # tout arrive en str : convertir soi-même
        print(f'{ligne["nom"]:<8} {prix:>7.2f} €   ({ligne["description"]})')

---
## 3. Le module `json` — le format d'échange universel

Le **JSON** (*JavaScript Object Notation*) est le format texte utilisé par la quasi-totalité
des APIs web et beaucoup de fichiers de configuration. Il ressemble énormément
aux dicts/listes Python :

| Python | JSON |
|---|---|
| `dict` | objet `{ }` |
| `list` | tableau `[ ]` |
| `str` | string `" "` (guillemets doubles obligatoires) |
| `int` / `float` | number |
| `True` / `False` / `None` | `true` / `false` / `null` |

Quatre fonctions à connaître — retiens : **s** comme **string**, sans s = fichier.

| Fonction | Sens |
|---|---|
| `json.dumps(objet)` | Python → texte JSON (**s** = vers une string) |
| `json.loads(texte)` | texte JSON → Python |
| `json.dump(objet, fichier)` | Python → fichier JSON |
| `json.load(fichier)` | fichier JSON → Python |

In [ ]:
import json
from pathlib import Path

config = {
    "application": "GestionStock",
    "version": "2.1",
    "debug": False,
    "seuils": {"alerte_stock": 5, "max_commande": 100},
    "entrepots": ["Bruxelles", "Liège", None],
}

# ── Python → texte JSON ──
texte = json.dumps(config, indent=2, ensure_ascii=False)
# indent=2 : mise en forme lisible | ensure_ascii=False : garde les accents tels quels
print(texte)
print(f"→ False est devenu {json.dumps(False)}, None est devenu {json.dumps(None)}")

# ── Sauvegarder / recharger dans un fichier ──
fichier_json = Path("demo_fichiers") / "config.json"
with open(fichier_json, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

with open(fichier_json, encoding="utf-8") as f:
    config_rechargee = json.load(f)

print(f"\nRelu depuis le fichier : {config_rechargee['seuils']['alerte_stock']} (type {type(config_rechargee['seuils']['alerte_stock']).__name__})")
print(f"Identique à l'original ? {config_rechargee == config}")

---
## 4. Cas pratique complet — convertir un CSV en JSON

Le combo classique en entreprise : recevoir un export CSV, le nettoyer, le convertir en JSON
pour une API. Ici tout ensemble : `pathlib` + `csv` + `json` + typage des valeurs.

In [ ]:
import csv
import json
from pathlib import Path

source = Path("demo_fichiers") / "produits.csv"       # créé à la section 2
destination = Path("demo_fichiers") / "produits.json"

# Lire le CSV et convertir les types (le CSV ne connaît que le texte)
with open(source, encoding="utf-8", newline="") as f:
    produits = [
        {"nom": ligne["nom"], "description": ligne["description"], "prix": float(ligne["prix"])}
        for ligne in csv.DictReader(f)
    ]

# Écrire le JSON
with open(destination, "w", encoding="utf-8") as f:
    json.dump(produits, f, indent=2, ensure_ascii=False)

print(destination.read_text(encoding="utf-8"))
print(f"→ les prix sont maintenant des nombres, prêts pour une API")

In [ ]:
# ── Nettoyage : supprimer les fichiers de démonstration ──
import shutil
from pathlib import Path

dossier = Path("demo_fichiers")
if dossier.exists():
    shutil.rmtree(dossier)      # supprime le dossier et tout son contenu
    print(f"Dossier {dossier}/ supprimé — notebook propre.")

---
## Récapitulatif

- `pathlib.Path` : chemins-objets, opérateur `/`, `read_text`/`write_text`, `glob` — l'API moderne.
- Toujours préciser `encoding="utf-8"` (le défaut varie selon la machine !).
- CSV : jamais de `split(",")` maison → `csv.DictReader`/`DictWriter` avec `newline=""`.
- JSON : `dumps`/`loads` pour du texte, `dump`/`load` pour un fichier ; `indent=2` et `ensure_ascii=False` pour rester lisible.
- Un CSV ne contient que du texte : convertir les nombres soi-même (`float(...)`).

> **En entretien** : « J'utilise `pathlib` plutôt qu'`os.path` : chemins portables Windows/Linux,
> API objet, et `glob` retourne un générateur donc pas de liste géante en mémoire pour parcourir
> un gros dossier. »